# **Information bottleneck attribution: карта в битах своим кодом**

Практика к модулю [«Объяснения от аксиом»](https://ai-interpretability.school/learn/12/0/0), урок «Обобщение по градиентным методам», шаг «Что общего у методов блока».

Information bottleneck attribution, или IBA ([Schulz et al., 2020](https://arxiv.org/abs/2001.00396)), определяет важность не через производную и не через разность с опорной точкой, а через пропускную способность: в один слой сети встраивается шум, и для каждой позиции подбирают, сколько информации о входе через нее надо пропустить, чтобы класс остался прежним. Здесь метод собран своим кодом на torch, примерно в тридцать строк, и сравнен с Integrated Gradients на том же снимке и той же модели, что в тетради [HW9_integrated_gradients](https://colab.research.google.com/github/SadSabrina/open-xai-materials/blob/main/homeworks/HW9_integrated_gradients.ipynb):

1. ResNet-50 и снимок кошки на кухне;
2. средние и разбросы активаций слоя по чужим снимкам ImageNet: без них шуму не из чего взяться;
3. сам bottleneck и карта в битах;
4. сравнение с картой IG: корреляция и доля массы в рамке кошки;
5. чувствительность к $\beta$, к выбору слоя, к классу и к сиду шума.

Считает около трех минут на процессоре.

In [ ]:
!pip install -q captum==0.9.0

In [ ]:
import io
import math
import tarfile
import time
import urllib.request
import warnings
warnings.filterwarnings('ignore', message='IProgress not found')   # tqdm без ipywidgets; в Colab ipywidgets есть

import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from PIL import Image
from torchvision import transforms
from torchvision.models import resnet50
from captum.attr import IntegratedGradients

torch.manual_seed(0)
DATA = 'https://raw.githubusercontent.com/SadSabrina/open-xai-materials/main'


def fetch(path):
    return urllib.request.urlopen(f'{DATA}/' + path, timeout=30).read()


model = resnet50(weights='IMAGENET1K_V1').eval()
for p in model.parameters():
    p.requires_grad_(False)

# та же предобработка, что в HW9_integrated_gradients
tf = transforms.Compose([transforms.Resize((224, 224)), transforms.ToTensor(),
                         transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])
photo = Image.open(io.BytesIO(fetch('data/cat.jpg'))).convert('RGB')
x = tf(photo).unsqueeze(0)
names = [s.strip() for s in fetch('data/imagenet_classes.txt').decode().split('\n')]

with torch.no_grad():
    probs = model(x).softmax(-1)[0]
for p, i in zip(*probs.topk(3)):
    print(f'{int(i):4d}  {names[i]:15}  {float(p):.3f}')

Модель уверенно видит кошку: класс `tiger cat` (282). Его и будем объяснять.

## 1. Средние и разбросы активаций

Шум в bottleneck не произвольный: он берется из того же распределения, что и активации слоя на обычных снимках, — для каждого канала и каждой позиции свои среднее $\mu$ и разброс $\sigma$. Тогда замена активации шумом стирает информацию о **нашем** снимке, но не выводит слой из привычного ему режима. Авторы оценивают $\mu$ и $\sigma$ по обучающей выборке ImageNet; здесь — по [Imagenette](https://github.com/fastai/imagenette), открытой выборке из десяти классов ImageNet от fast.ai. Архив читается потоком: проверочная часть лежит в нем первой, из нее берется каждый двадцатый снимок, всего 197, и чтение останавливается, не дойдя до обучающей части.

Моменты считаются сразу для трех слоев ResNet-50: `layer1`, `layer2` и `layer3`. Основной слой — `layer2`, выход блока conv3 размером $512\times28\times28$: на нем авторы показывают свои карты для ResNet-50.

In [ ]:
LAYERS = {'layer1': model.layer1, 'layer2': model.layer2, 'layer3': model.layer3}
URL = 'https://s3.amazonaws.com/fast-ai-imageclas/imagenette2-160.tgz'

start = time.time()
images = []
with urllib.request.urlopen(URL, timeout=60) as stream:
    tar = tarfile.open(fileobj=stream, mode='r|gz')
    k = 0
    for member in tar:
        if '/train/' in member.name:
            break                                   # проверочная часть кончилась, дальше не читаем
        if member.name.endswith('.JPEG'):
            if k % 20 == 0:                         # каждый двадцатый снимок
                images.append(tf(Image.open(tar.extractfile(member)).convert('RGB')))
            k += 1
print(f'снимков в проверочной части: {k}, взято: {len(images)}, {time.time() - start:.0f} с')

# суммы активаций и их квадратов по снимкам, в double, чтобы разность E[a^2] - E[a]^2 не теряла точность
sums = {name: [0.0, 0.0] for name in LAYERS}
feats = {}
hooks = [m.register_forward_hook(lambda mod, inp, out, name=name: feats.__setitem__(name, out.double()))
         for name, m in LAYERS.items()]
with torch.no_grad():
    for i in range(0, len(images), 32):
        model(torch.stack(images[i:i + 32]))
        for name, a in feats.items():
            sums[name] = [sums[name][0] + a.sum(0), sums[name][1] + (a ** 2).sum(0)]
for h in hooks:
    h.remove()

N = len(images)
MU = {name: (s / N).float() for name, (s, s2) in sums.items()}
# разброс снизу ограничен 0.01, как в коде авторов: иначе у всегда нулевых после ReLU активаций деление на ноль
STD = {name: (s2 / N - (s / N) ** 2).clamp(min=0).sqrt().float().clamp(min=0.01) for name, (s, s2) in sums.items()}
for name in LAYERS:
    print(f'{name}: форма активаций {tuple(MU[name].shape)}')

## 2. Bottleneck

Пусть $R$ — активации выбранного слоя на нашем снимке. Bottleneck заменяет их смесью с шумом:

$$Z=\lambda\odot R+(1-\lambda)\odot\varepsilon,\qquad \varepsilon\sim\mathcal N(\mu,\sigma^2),\qquad \lambda=\sigma_{\text{sig}}(\alpha)\in(0,1),$$

где $\lambda$ — своя доля для каждого канала и каждой позиции. При $\lambda=1$ активация проходит целиком, при $\lambda=0$ от снимка не остается ничего, только шум. Сколько информации о $R$ осталось в $Z$, оценивается расхождением Кульбака–Лейблера между распределением $Z$ при данном $R$ и распределением шума. В нормированных координатах $r=(R-\mu)/\sigma$ это гауссово расхождение считается в одну строку:

$$I_{ij}=\tfrac12\left(\lambda_{ij}^2r_{ij}^2+(1-\lambda_{ij})^2-1\right)-\log(1-\lambda_{ij}).$$

Параметры $\alpha$ подбираются под один снимок оптимизацией функции

$$\mathcal L=\mathrm{CE}\big(f(Z),\,c\big)+\beta\cdot\overline{I},$$

где первое слагаемое требует, чтобы класс $c$ остался прежним, а второе, среднее $I$ по всем элементам слоя, штрафует за каждый пропущенный бит. Настройки — как у авторов: $\alpha$ стартует с 5 ($\lambda\approx0.993$, почти все проходит), 10 шагов Adam с шагом 1, на каждом шаге 10 копий снимка со своим шумом, $\beta=10$, $\lambda$ сглажена гауссовым ядром с $\sigma=1$ по позициям. Карта — сумма $I$ по каналам, переведенная в биты делением на $\ln 2$.

In [ ]:
def gaussian_kernel(sigma=1.0):
    # ядро 7x7, как у авторов при sigma = 1
    r = torch.arange(-3, 4, dtype=torch.float32)
    g = torch.exp(-r ** 2 / (2 * sigma ** 2))
    g = g / g.sum()
    return g[:, None] * g[None, :]


KERNEL = gaussian_kernel()


def smooth(lam):
    # сглаживание доли lambda по позициям, отдельно в каждом канале
    c = lam.shape[1]
    return F.conv2d(F.pad(lam, (3, 3, 3, 3), mode='reflect'), KERNEL.expand(c, 1, 7, 7), groups=c)


def iba(x, target, layer='layer2', beta=10.0, steps=10, batch=10, lr=1.0, seed=0):
    # карта в битах: сколько информации о входе каждая позиция слоя пропускает, чтобы класс target остался
    mu, std = MU[layer], STD[layer]
    gen = torch.Generator().manual_seed(seed)
    alpha = torch.full((1,) + tuple(mu.shape), 5.0, requires_grad=True)
    opt = torch.optim.Adam([alpha], lr=lr)
    state = {}

    def bottleneck(module, inputs, R):
        lam = smooth(torch.sigmoid(alpha))
        eps = mu + std * torch.randn(R.shape, generator=gen)
        r = (R - mu) / std
        state['I'] = 0.5 * (lam ** 2 * r ** 2 + (1 - lam) ** 2 - 1) - torch.log(1 - lam)
        return lam * R + (1 - lam) * eps

    hook = LAYERS[layer].register_forward_hook(bottleneck)
    xs = x.expand(batch, -1, -1, -1)
    labels = torch.full((batch,), target)
    for _ in range(steps):
        opt.zero_grad()
        loss = F.cross_entropy(model(xs), labels) + beta * state['I'].mean()
        loss.backward()
        opt.step()
    with torch.no_grad():
        model(x)                                   # последний проход: I при итоговых lambda
    hook.remove()
    return state['I'][0].sum(0) / math.log(2)


TARGET = 282
start = time.time()
bits = iba(x, TARGET)
print(f'время: {time.time() - start:.1f} с, карта {tuple(bits.shape)}')
print(f'всего пропущено: {float(bits.sum()):.0f} бит, максимум в одной позиции: {float(bits.max()):.0f} бит')

## 3. Сравнение с Integrated Gradients

Карта IG для того же класса: `captum.attr.IntegratedGradients`, нулевой baseline, 50 шагов, как в HW9. Чтобы сравнивать на одной решетке, модули атрибуций IG суммируются по каналам и усредняются в квадратах $8\times8$ пикселей — получается та же решетка $28\times28$, что у слоя `layer2`.

Два числа сравнения: корреляция Пирсона двух карт и доля массы карты, попавшая в рамку кошки. Рамка размечена на глаз по снимку $224\times224$ и занимает около десятой части кадра: карта, размазанная по кадру равномерно, положила бы в нее столько же.

In [ ]:
ig = IntegratedGradients(model).attribute(x, baselines=x * 0, target=TARGET, n_steps=50, internal_batch_size=10)
ig_map = F.avg_pool2d(ig.abs().sum(1, keepdim=True), 8)[0, 0]      # 224x224 -> 28x28

CAT_BOX = (90, 55, 132, 180)          # x0, y0, x1, y1 на снимке 224x224, размечено на глаз


def corr(a, b):
    a, b = a.flatten() - a.mean(), b.flatten() - b.mean()
    return float((a * b).sum() / (a.norm() * b.norm()))


def mass_in(m, box=CAT_BOX):
    # доля массы карты внутри рамки; карта растянута до 224x224
    big = F.interpolate(m[None, None], (224, 224), mode='bilinear', align_corners=False)[0, 0]
    x0, y0, x1, y1 = box
    return float(big[y0:y1, x0:x1].sum() / big.sum())


area = (CAT_BOX[2] - CAT_BOX[0]) * (CAT_BOX[3] - CAT_BOX[1]) / 224 ** 2
print(f'рамка кошки занимает {area * 100:.1f}% кадра')
print(f'доля массы в рамке кошки: IBA {mass_in(bits) * 100:.1f}%, IG {mass_in(ig_map) * 100:.1f}%')
print(f'корреляция карт IBA и IG: {corr(bits, ig_map):.3f}')

IBA кладет в рамку кошки в несколько раз больше массы, чем IG, и в несколько раз больше, чем досталось бы рамке при равномерной карте. Корреляция двух карт при этом невысокая: IG раздает заметную долю атрибуции фону кухни, а IBA почти весь фон глушит шумом, и класс от этого не меняется.

Причина в том, о чем спрашивает каждый метод. IG отвечает, сколько каждый пиксель добавил к логиту по пути от черного снимка; фон тоже лежит на этом пути, и его вклад не нулевой. IBA отвечает, без чего класс не удержать: фон модели не нужен, если про кошку информация прошла.

## 4. Цена информации $\beta$

$\beta$ — единственный содержательный параметр метода. Чем он больше, тем дороже каждый бит, и тем меньше информации bottleneck пропускает.

In [ ]:
print(f'{"beta":>6}{"всего бит":>12}{"в рамке кошки":>16}{"корр. с IG":>13}')
for beta in (1, 10, 100):
    m = iba(x, TARGET, beta=beta)
    print(f'{beta:>6}{float(m.sum()):>12.0f}{mass_in(m) * 100:>15.1f}%{corr(m, ig_map):>13.3f}')

С ростом $\beta$ всего пропущенных бит становится меньше, а доля их в рамке кошки растет: bottleneck оставляет только самое необходимое. Карта в битах поэтому не абсолютная величина: число бит и форма карты объявляются вместе с $\beta$.

## 5. Выбор слоя

Bottleneck можно встроить в любой слой. Ранний слой видит мелкие детали, и решетка у него частая: у `layer1` она $56\times56$. У позднего `layer3` решетка $14\times14$, зато каждая позиция в нем уже знает о большом куске снимка.

In [ ]:
layer_maps = {}
print(f'{"слой":>8}{"решетка":>10}{"всего бит":>12}{"в рамке кошки":>16}')
for name in LAYERS:
    layer_maps[name] = iba(x, TARGET, layer=name)
    m = layer_maps[name]
    print(f'{name:>8}{m.shape[0]:>7}x{m.shape[1]:<2}{float(m.sum()):>12.0f}{mass_in(m) * 100:>15.1f}%')

Слой меняет и решетку карты, и то, сколько массы попадает на кошку. Как и у LRP с его правилами, слой объявляется вместе с картой.

## 6. Про какой класс спросили

IBA, в отличие от сырой карты внимания, зависит от класса: штраф за биты один и тот же, а удержать надо другой ответ.

**Задание.** Постройте карту IBA на слое `layer2` при $\beta=10$ для класса `tabby` (281, тоже кошка) и для класса `dining table` (532, обеденный стол, на котором кошка сидит). Для каждой выведите, сколько всего бит пропущено, долю массы в рамке кошки и корреляцию с картой для `tiger cat`. Какой из двух классов потребовал пропустить больше информации и куда ушла масса карты?

In [ ]:
# Ваш код здесь

## 7. Карта зависит от сида шума

Bottleneck оптимизируется по случайному шуму, поэтому два запуска с разным сидом дают разные карты. Насколько разные — проверим на трех сидах.

In [ ]:
print(f'{"сид":>4}{"всего бит":>12}{"в рамке кошки":>16}{"корр. с сидом 0":>18}')
for seed in (0, 1, 2):
    m = iba(x, TARGET, seed=seed)
    print(f'{seed:>4}{float(m.sum()):>12.0f}{mass_in(m) * 100:>15.1f}%{corr(m, bits):>18.4f}')

Форма карты от сида почти не зависит: корреляция с картой сида 0 выше 0.99. Сумма пропущенных бит при этом различается между сидами больше чем на десятую часть, а доля массы в рамке кошки — на несколько процентных пунктов. Это цена оптимизации на каждый объект: у IG при том же числе шагов результат один и тот же при каждом запуске, а у IBA сид шума входит в описание карты наравне с $\beta$ и слоем. Ту же оптимизацию на GPU генератор шума проведет по другим случайным числам, и числа этой тетради там сойдутся только приблизительно.

## 8. Посмотреть глазами

In [ ]:
panels = [('photo', None), ('IBA, layer2, beta = 10', bits), ('IG, zero baseline', ig_map),
          ('IBA, layer3', layer_maps['layer3']), ('IBA, class dining table', iba(x, 532))]
fig, axes = plt.subplots(1, 5, figsize=(18, 4))
for ax, (title, m) in zip(axes, panels):
    ax.imshow(photo.resize((224, 224)))
    if m is not None:
        big = F.interpolate(m[None, None], (224, 224), mode='bilinear', align_corners=False)[0, 0]
        ax.imshow(big.numpy(), cmap='inferno', alpha=0.7)
    x0, y0, x1, y1 = CAT_BOX
    ax.add_patch(plt.Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, color='cyan', lw=1.5))
    ax.set_title(title)
    ax.axis('off')
plt.tight_layout()
plt.show()

## Что унести из тетради

- **IBA спрашивает, без чего класс не удержать.** Позиция важна, если заменить ее шумом нельзя без потери ответа; фон, без которого модель обходится, получает почти ноль бит, даже когда IG раздает ему заметную атрибуцию.
- **Шум берется из распределения активаций слоя.** Без средних и разбросов по чужим снимкам метод не определен; здесь их дала Imagenette, у авторов — обучающая выборка ImageNet.
- **Карта объявляется вместе с $\beta$, слоем и сидом шума.** $\beta$ меняет и число бит, и долю на объекте, слой — решетку и форму карты, сид — сумму бит и долю массы в рамке.
- **Суммы карты с разностью прогнозов не сравнивают.** Карта в битах полноте не подчиняется: в этом IBA отличается от IG, DeepLIFT и LRP.
- **Оптимизация на каждый объект.** Десять шагов оптимизации, на каждом прямой и обратный проход по десяти копиям снимка: сто проходов снимка туда и обратно на одну карту, не считая оценки моментов. IG с 50 шагами обходится пятьюдесятью.